In [3]:
import json
import glob
import os
import pandas as pd

cache_dir = 'hcdp_station_cache'
files = sorted(glob.glob(os.path.join(cache_dir, '*_all_rf.json')))

rows = []
for fpath in files:
    with open(fpath, 'r') as f:
        data = json.load(f)
    station_id = data.get('station_id', os.path.basename(fpath))
    station_name = data.get('station_name', '')
    for var_name, var_data in data.get('variables', {}).items():
        records = var_data.get('records', [])
        if not records:
            rows.append({'station_id': station_id, 'station_name': station_name,
                         'variable': var_name, 'record_count': 0,
                         'earliest': None, 'latest': None})
            continue
        timestamps = [r['timestamp'] for r in records]
        earliest = min(timestamps)
        latest = max(timestamps)
        rows.append({
            'station_id': station_id,
            'station_name': station_name,
            'variable': var_name,
            'record_count': var_data.get('record_count', len(records)),
            'earliest': earliest,
            'latest': latest,
        })

df = pd.DataFrame(rows)
df['earliest'] = pd.to_datetime(df['earliest'], utc=True)
df['latest'] = pd.to_datetime(df['latest'], utc=True)
df['span_days'] = (df['latest'] - df['earliest']).dt.days

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
display(df.sort_values('station_id').reset_index(drop=True))

,station_id,station_name,variable,record_count,earliest,latest,span_days
0,0115,Piiholo,RF_1_Tot300s,205630,2022-12-01 22:40:00+00:00,2025-01-01 09:55:00+00:00,761
1,0115,Piiholo,RF_1_Tot60s,12026,2022-12-02 00:17:00+00:00,2025-01-01 09:32:00+00:00,761
2,0116,Keokea,RF_1_Tot300s,218901,2022-12-01 01:30:00+00:00,2025-01-01 09:55:00+00:00,762
3,0116,Keokea,RF_1_Tot60s,4508,2022-12-02 23:50:00+00:00,2024-11-06 06:51:00+00:00,704
4,0118,Pulehu,RF_1_Tot300s,106137,2023-12-28 22:30:00+00:00,2025-01-01 09:55:00+00:00,369
5,0118,Pulehu,RF_1_Tot60s,1268,2023-12-28 22:28:00+00:00,2024-11-14 10:52:00+00:00,321
6,0119,Kula Ag,RF_1_Tot300s,284562,2022-04-15 20:50:00+00:00,2025-01-01 09:55:00+00:00,991
7,0119,Kula Ag,RF_1_Tot60s,4525,2022-04-17 00:04:00+00:00,2024-11-06 05:20:00+00:00,934
8,0121,Lipoa,RF_1_Tot300s,112772,2023-12-05 23:20:00+00:00,2025-01-01 09:55:00+00:00,392
9,0121,Lipoa,RF_1_Tot60s,831,2023-12-14 07:34:00+00:00,2024-10-27 23:29:00+00:00,318
